# Punto 5 — Dashboard del partido

**Proyecto Final · Curso LanusStats** · Lucas Marinelli · @datafutbol_ar

> **Consigna oficial.** Del partido hacer un dashboard que contenga:
> 1. Mapa de tiros y mapas de pases de ambos equipos
> 2. Resultado y detalles del partido
> 3. Datos sobre jugadores destacados (criterio propio: más recuperaciones, pases clave, etc.)

Partido: **Argentina 1–2 Arabia Saudita** — debut Mundial 2022.

### Cómo lo organicé

Hice **dos dashboards** complementarios para cubrir bien la consigna:

| | Dashboard A (académico) | Dashboard B (jugador destacado) |
|---|---|---|
| **Cumple** | Mapas de tiros + pases ambos equipos + resultado + tabla métricas + sección "destacados" | "Datos sobre jugadores destacados" con foco en Messi |
| **Audiencia** | Curso | IG/X (público @datafutbol_ar) |
| **Formato** | 1080×1620 px (carrusel IG largo) | 1080×1350 px (carrusel IG estándar) |
| **Archivo** | `punto5_dashboard_partido.png` | `punto5_dashboard_messi.png` |


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from helpers import *
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mplsoccer import Pitch, VerticalPitch
from pases_progresivos import agregar_pases_progresivos

# Paleta semáforo CVD-safe (Wong 2011)
COLOR_OK     = '#009E73'
COLOR_NO_OK  = '#B36930'

# Colormap "datafutbol" para heatmaps
cmap_marca = LinearSegmentedColormap.from_list(
    'datafutbol', [COLORS['bg'], COLORS['primary'], COLORS['accent']]
)

ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')
ev = añadir_xy(ev)
print(f'eventos: {ev.shape}')


## 1. Calcular stats agregadas por equipo

In [ ]:
def stats_equipo(ev, equipo):
    """Métricas agregadas de un equipo en el partido."""
    ev_eq = ev[ev['team'] == equipo]

    tiros = ev_eq[ev_eq['type'] == 'Shot']
    goles = int((tiros['shot_outcome'] == 'Goal').sum())
    al_arco = int(tiros['shot_outcome'].isin(['Goal', 'Saved', 'Saved To Post']).sum())
    xg_total = round(tiros['shot_statsbomb_xg'].sum(), 2)

    pases = ev_eq[ev_eq['type'] == 'Pass']
    pases_ok = int(pases['pass_outcome'].isna().sum())
    pct_acierto = round(pases_ok / len(pases) * 100, 1) if len(pases) else 0

    # Pases progresivos con la función del módulo
    pases_ext = agregar_pases_progresivos(pases)
    n_prog = int(pases_ext['es_progresivo'].sum())

    recup = int((ev_eq['type'] == 'Ball Recovery').sum())
    posesion = round(len(ev_eq) / len(ev) * 100, 1)

    return {
        'equipo': equipo,
        'goles': goles, 'tiros': len(tiros), 'tiros_al_arco': al_arco,
        'xg': xg_total, 'pases_total': len(pases), 'pases_ok': pases_ok,
        'acierto_pct': pct_acierto, 'pases_progresivos': n_prog,
        'recuperaciones': recup, 'posesion': posesion,
    }


stats_arg = stats_equipo(ev, 'Argentina')
stats_sau = stats_equipo(ev, 'Saudi Arabia')

for k, v in stats_arg.items(): print(f'  ARG · {k}: {v}')
print()
for k, v in stats_sau.items(): print(f'  SAU · {k}: {v}')


## 2. Calcular jugadores destacados de cada equipo

**Mi criterio de "destacado":** el jugador con más **pases progresivos** completados de cada equipo. Esa métrica captura mejor "quién hizo avanzar el juego" que el simple conteo de pases o recuperaciones.


In [ ]:
def jugador_destacado(ev, equipo):
    """Devuelve el jugador con más pases progresivos del equipo."""
    pases = ev[(ev['type'] == 'Pass') & (ev['team'] == equipo)]
    pases_ext = agregar_pases_progresivos(pases)
    prog = pases_ext[pases_ext['es_progresivo']]
    if prog.empty:
        return None
    top = prog.groupby('player').size().sort_values(ascending=False)
    nombre = top.index[0]
    n_prog = int(top.iloc[0])

    # Stats adicionales del destacado
    ev_jug = ev[ev['player'] == nombre]
    n_pases = int((ev_jug['type'] == 'Pass').sum())
    n_recup = int((ev_jug['type'] == 'Ball Recovery').sum())
    n_tiros = int((ev_jug['type'] == 'Shot').sum())
    return {'nombre': nombre, 'equipo': equipo,
            'pases_progresivos': n_prog, 'pases_totales': n_pases,
            'recuperaciones': n_recup, 'tiros': n_tiros}


destacado_arg = jugador_destacado(ev, 'Argentina')
destacado_sau = jugador_destacado(ev, 'Saudi Arabia')

print('Jugador destacado · Argentina:')
print(f'  {destacado_arg}')
print()
print('Jugador destacado · Saudi Arabia:')
print(f'  {destacado_sau}')


---

## 3. Dashboard A — Partido completo (académico)

Layout en grilla 1080×1620 px (formato carrusel IG largo):

```
+--------------------------------+
| HEADER (resultado + título)    |
+----------------+----------------+
| Shot map ARG   | Shot map SAU   |   ← Mapas de tiros (consigna 5.1a)
+----------------+----------------+
| Pass map ARG   | Pass map SAU   |   ← Mapas de pases (consigna 5.1b)
+--------------------------------+
| TABLA COMPARATIVA              |   ← Resultado y detalles (consigna 5.2)
+--------------------------------+
| JUGADORES DESTACADOS           |   ← Criterio propio (consigna 5.3)
+--------------------------------+
| Footer                          |
+--------------------------------+
```


In [ ]:
def dashboard_partido(ev, stats_a, stats_b, dest_a, dest_b, archivo):
    """Dashboard A: comparativa completa del partido en 1080×1620."""
    fig = plt.figure(figsize=(10.8, 16.2), dpi=100, facecolor=COLORS['bg'])
    gs = gridspec.GridSpec(
        nrows=6, ncols=2,
        height_ratios=[1.0, 5.0, 4.0, 3.2, 2.4, 0.4],
        hspace=0.30, wspace=0.10,
        left=0.05, right=0.95, top=0.97, bottom=0.03
    )

    # ===== HEADER =====
    ax_head = fig.add_subplot(gs[0, :]); ax_head.axis('off')
    ax_head.text(0.5, 0.85, 'Argentina vs Arabia Saudita',
                 ha='center', va='top', color=COLORS['text'],
                 fontsize=22, weight='bold', transform=ax_head.transAxes)
    ax_head.text(0.5, 0.55, 'Debut Mundial 2022 · Lusail Stadium',
                 ha='center', va='top', color=COLORS['accent'],
                 fontsize=13, style='italic', transform=ax_head.transAxes)
    resultado = f"{stats_a['goles']}  –  {stats_b['goles']}"
    ax_head.text(0.5, 0.20, resultado, ha='center', va='top',
                 color=COLORS['text'], fontsize=36, weight='bold',
                 family='monospace', transform=ax_head.transAxes)

    # ===== FILA 1: SHOT MAPS =====
    def _shot_map(ax, equipo, color_resaltado):
        tiros = ev[(ev['type'] == 'Shot') & (ev['team'] == equipo)].copy()
        pitch = VerticalPitch(pitch_type='statsbomb', pitch_color=COLORS['bg'],
                              line_color=COLORS['text'], linewidth=1.2,
                              half=True, pad_top=2)
        pitch.draw(ax=ax); ax.set_facecolor(COLORS['bg'])
        es_gol = tiros['shot_outcome'] == 'Goal'
        size = tiros['shot_statsbomb_xg'] * 500 + 80
        pitch.scatter(tiros.loc[~es_gol, 'x'], tiros.loc[~es_gol, 'y'],
                       s=size[~es_gol], ax=ax, color=COLORS['primary'],
                       edgecolors=COLORS['text'], linewidth=1.2,
                       alpha=0.85, zorder=2)
        if es_gol.any():
            pitch.scatter(tiros.loc[es_gol, 'x'], tiros.loc[es_gol, 'y'],
                           s=size[es_gol], ax=ax, color=COLORS['accent'],
                           edgecolors=COLORS['text'], linewidth=2,
                           alpha=0.95, zorder=3)
        ax.set_title(f'{equipo.upper()} · TIROS',
                     color=color_resaltado, fontsize=13, weight='bold', pad=8)

    _shot_map(fig.add_subplot(gs[1, 0]), 'Argentina', COLORS['primary'])
    _shot_map(fig.add_subplot(gs[1, 1]), 'Saudi Arabia', COLORS['accent'])

    # ===== FILA 2: PASS MAPS (NUEVA — cumple consigna 5.1b) =====
    def _pass_map(ax, equipo, color_resaltado):
        pases = ev[(ev['type'] == 'Pass') & (ev['team'] == equipo)].copy()
        pases = agregar_pases_progresivos(pases)
        pitch = Pitch(pitch_type='statsbomb', pitch_color=COLORS['bg'],
                      line_color=COLORS['text'], linewidth=1.0, pad_top=8)
        pitch.draw(ax=ax); ax.set_facecolor(COLORS['bg'])
        ok = pases['es_completado']
        pitch.arrows(pases.loc[ok, 'x'], pases.loc[ok, 'y'],
                     pases.loc[ok, 'x_end'], pases.loc[ok, 'y_end'],
                     ax=ax, width=0.8, headwidth=3, color=COLOR_OK,
                     alpha=0.45, zorder=2)
        pitch.arrows(pases.loc[~ok, 'x'], pases.loc[~ok, 'y'],
                     pases.loc[~ok, 'x_end'], pases.loc[~ok, 'y_end'],
                     ax=ax, width=0.8, headwidth=3, color=COLOR_NO_OK,
                     alpha=0.55, zorder=3)
        ax.set_title(f'{equipo.upper()} · PASES ({int(ok.sum())} ok / {len(pases)} total)',
                     color=color_resaltado, fontsize=11, weight='bold', pad=6)

    _pass_map(fig.add_subplot(gs[2, 0]), 'Argentina', COLORS['primary'])
    _pass_map(fig.add_subplot(gs[2, 1]), 'Saudi Arabia', COLORS['accent'])

    # ===== TABLA COMPARATIVA =====
    ax_tab = fig.add_subplot(gs[3, :]); ax_tab.axis('off')
    metricas = [
        ('Goles', stats_a['goles'], stats_b['goles']),
        ('Tiros', stats_a['tiros'], stats_b['tiros']),
        ('Tiros al arco', stats_a['tiros_al_arco'], stats_b['tiros_al_arco']),
        ('xG', stats_a['xg'], stats_b['xg']),
        ('Pases progresivos', stats_a['pases_progresivos'], stats_b['pases_progresivos']),
        ('% acierto pases', f"{stats_a['acierto_pct']}%", f"{stats_b['acierto_pct']}%"),
        ('Recuperaciones', stats_a['recuperaciones'], stats_b['recuperaciones']),
        ('Posesión (proxy)', f"{stats_a['posesion']}%", f"{stats_b['posesion']}%"),
    ]
    n = len(metricas); y_top = 0.95; y_bot = 0.05
    row_h = (y_top - y_bot) / (n + 1)

    ax_tab.text(0.20, y_top - row_h*0.5, 'ARGENTINA', ha='center', va='center',
                color=COLORS['primary'], fontsize=12, weight='bold',
                transform=ax_tab.transAxes)
    ax_tab.text(0.50, y_top - row_h*0.5, 'MÉTRICA', ha='center', va='center',
                color=COLORS['muted'], fontsize=11, style='italic',
                transform=ax_tab.transAxes)
    ax_tab.text(0.80, y_top - row_h*0.5, 'ARABIA', ha='center', va='center',
                color=COLORS['accent'], fontsize=12, weight='bold',
                transform=ax_tab.transAxes)
    ax_tab.plot([0.05, 0.95], [y_top - row_h, y_top - row_h],
                color=COLORS['accent'], linewidth=1.2, alpha=0.6,
                transform=ax_tab.transAxes)

    for i, (nombre, val_a, val_b) in enumerate(metricas):
        y = y_top - row_h * (i + 1.7)
        try:
            num_a = float(str(val_a).rstrip('%'))
            num_b = float(str(val_b).rstrip('%'))
            color_a = COLORS['primary'] if num_a >= num_b else COLORS['muted']
            color_b = COLORS['accent']  if num_b >= num_a else COLORS['muted']
            peso_a = 'bold' if num_a >= num_b else 'normal'
            peso_b = 'bold' if num_b >= num_a else 'normal'
        except ValueError:
            color_a = color_b = COLORS['text']; peso_a = peso_b = 'normal'
        ax_tab.text(0.20, y, str(val_a), ha='center', va='center',
                    color=color_a, fontsize=13, weight=peso_a,
                    family='monospace', transform=ax_tab.transAxes)
        ax_tab.text(0.50, y, nombre, ha='center', va='center',
                    color=COLORS['text'], fontsize=10,
                    transform=ax_tab.transAxes)
        ax_tab.text(0.80, y, str(val_b), ha='center', va='center',
                    color=color_b, fontsize=13, weight=peso_b,
                    family='monospace', transform=ax_tab.transAxes)

    # ===== JUGADORES DESTACADOS (NUEVA — cumple consigna 5.3) =====
    ax_dest = fig.add_subplot(gs[4, :]); ax_dest.axis('off')
    ax_dest.text(0.5, 0.95, 'JUGADORES DESTACADOS · más pases progresivos por equipo',
                 ha='center', va='top', color=COLORS['text'],
                 fontsize=13, weight='bold', transform=ax_dest.transAxes)

    def _card(ax, x_center, dest, color):
        if dest is None: return
        # Recuadro
        box = FancyBboxPatch(
            (x_center - 0.22, 0.10), 0.44, 0.65,
            boxstyle="round,pad=0.01,rounding_size=0.02",
            facecolor=COLORS['bg'], edgecolor=color,
            linewidth=1.5, transform=ax.transAxes
        )
        ax.add_patch(box)
        # Nombre + equipo
        nombre_corto = dest['nombre'].split()
        nombre_corto = f"{nombre_corto[0]} {nombre_corto[-1]}" if len(nombre_corto) >= 2 else dest['nombre']
        ax.text(x_center, 0.68, nombre_corto,
                ha='center', va='top', color=color,
                fontsize=12, weight='bold', transform=ax.transAxes)
        ax.text(x_center, 0.56, dest['equipo'],
                ha='center', va='top', color=COLORS['muted'],
                fontsize=9, style='italic', transform=ax.transAxes)
        # Métricas
        ax.text(x_center - 0.12, 0.40, f"{dest['pases_progresivos']}",
                ha='center', va='top', color=COLORS['text'],
                fontsize=18, weight='bold', family='monospace',
                transform=ax.transAxes)
        ax.text(x_center - 0.12, 0.22, "PASES PROG.",
                ha='center', va='top', color=COLORS['muted'],
                fontsize=8, transform=ax.transAxes)
        ax.text(x_center + 0.12, 0.40, f"{dest['recuperaciones']}",
                ha='center', va='top', color=COLORS['text'],
                fontsize=18, weight='bold', family='monospace',
                transform=ax.transAxes)
        ax.text(x_center + 0.12, 0.22, "RECUPERAC.",
                ha='center', va='top', color=COLORS['muted'],
                fontsize=8, transform=ax.transAxes)

    _card(ax_dest, 0.27, dest_a, COLORS['primary'])
    _card(ax_dest, 0.73, dest_b, COLORS['accent'])

    # ===== FOOTER =====
    ax_foot = fig.add_subplot(gs[5, :]); ax_foot.axis('off')
    ax_foot.text(0.02, 0.5, '@datafutbol_ar', ha='left', va='center',
                 color=COLORS['accent'], fontsize=10, weight='bold',
                 transform=ax_foot.transAxes)
    ax_foot.text(0.98, 0.5, 'Datos: StatsBomb Open Data', ha='right', va='center',
                 color=COLORS['muted'], fontsize=9, style='italic',
                 transform=ax_foot.transAxes)

    guardar_fig(fig, archivo)
    plt.show()


dashboard_partido(ev, stats_arg, stats_sau, destacado_arg, destacado_sau,
                  'punto5_dashboard_partido')


---

## 4. Dashboard B — Foco en el jugador destacado (Messi)

Mismo formato 1080×1350 pero centrado en Messi como **jugador destacado del partido** según mi criterio editorial (capitán + único goleador de ARG + más relevante para la narrativa).


In [ ]:
def dashboard_jugador(ev, jugador, archivo, equipo_jugador='Argentina'):
    """Dashboard B: foco individual."""
    ev_jug = ev[ev['player'] == jugador].copy()

    tiros = ev_jug[ev_jug['type'] == 'Shot']
    goles = (tiros['shot_outcome'] == 'Goal').sum()
    xg = tiros['shot_statsbomb_xg'].sum()
    pases = ev_jug[ev_jug['type'] == 'Pass']
    pases_ok = pases['pass_outcome'].isna().sum()
    acierto = pases_ok / len(pases) * 100 if len(pases) else 0
    pases_ext = agregar_pases_progresivos(pases)
    n_prog = int(pases_ext['es_progresivo'].sum())

    fig = plt.figure(figsize=(10.8, 13.5), dpi=100, facecolor=COLORS['bg'])
    gs = gridspec.GridSpec(
        nrows=5, ncols=2,
        height_ratios=[1.3, 4.5, 5, 1.8, 0.5],
        hspace=0.35, wspace=0.15,
        left=0.06, right=0.94, top=0.96, bottom=0.04
    )

    ax_head = fig.add_subplot(gs[0, :]); ax_head.axis('off')
    ax_head.text(0.5, 0.85, 'LIONEL MESSI' if 'Messi' in jugador else jugador.upper(),
                 ha='center', va='top', color=COLORS['text'],
                 fontsize=26, weight='bold', transform=ax_head.transAxes)
    ax_head.text(0.5, 0.45, '#10 · Capitán · Argentina',
                 ha='center', va='top', color=COLORS['accent'],
                 fontsize=14, style='italic', transform=ax_head.transAxes)
    ax_head.text(0.5, 0.10, 'vs Arabia Saudita · Mundial 2022',
                 ha='center', va='top', color=COLORS['muted'],
                 fontsize=11, transform=ax_head.transAxes)

    ax_heat = fig.add_subplot(gs[1, :])
    pos = ev_jug.dropna(subset=['x', 'y'])
    pitch_h = Pitch(pitch_type='statsbomb', pitch_color=COLORS['bg'],
                    line_color=COLORS['text'], linewidth=1.3,
                    line_zorder=2, pad_top=10)
    pitch_h.draw(ax=ax_heat)
    bin_stat = pitch_h.bin_statistic(pos.x, pos.y, statistic='count', bins=(12, 8))
    pitch_h.heatmap(bin_stat, ax=ax_heat, cmap=cmap_marca,
                     edgecolors=COLORS['bg'], alpha=0.85, zorder=1)
    ax_heat.set_title('MAPA DE CALOR', color=COLORS['accent'],
                       fontsize=13, weight='bold', loc='left', pad=8)

    ax_shot = fig.add_subplot(gs[2, 0])
    pitch_v = VerticalPitch(pitch_type='statsbomb', pitch_color=COLORS['bg'],
                            line_color=COLORS['text'], linewidth=1.3,
                            half=True, pad_top=2)
    pitch_v.draw(ax=ax_shot)
    es_gol = tiros['shot_outcome'] == 'Goal'
    size = tiros['shot_statsbomb_xg'] * 500 + 80
    pitch_v.scatter(tiros.loc[~es_gol, 'x'], tiros.loc[~es_gol, 'y'],
                     s=size[~es_gol], ax=ax_shot, color=COLORS['primary'],
                     edgecolors=COLORS['text'], linewidth=1.3, alpha=0.85)
    if es_gol.any():
        pitch_v.scatter(tiros.loc[es_gol, 'x'], tiros.loc[es_gol, 'y'],
                         s=size[es_gol], ax=ax_shot, color=COLORS['accent'],
                         edgecolors=COLORS['text'], linewidth=2, alpha=0.95)
    ax_shot.set_title('MAPA DE TIROS', color=COLORS['accent'],
                       fontsize=13, weight='bold', loc='left', pad=8)

    ax_kpi = fig.add_subplot(gs[2, 1]); ax_kpi.axis('off')
    kpis = [
        (f'{len(tiros)}', 'TIROS', f'{int(goles)} GOL'),
        (f'{xg:.2f}', 'xG TOTAL', 'expected goals'),
        (f'{int(n_prog)}', 'PASES PROGRESIVOS', 'avance >=10m'),
        (f'{int(acierto)}%', 'ACIERTO PASES', f'{int(pases_ok)}/{len(pases)}'),
    ]
    for i, (num, label, sub) in enumerate(kpis):
        y_top = 1 - i * 0.25
        box = FancyBboxPatch(
            (0.05, y_top - 0.22), 0.90, 0.20,
            boxstyle="round,pad=0.01,rounding_size=0.02",
            facecolor=COLORS['bg'], edgecolor=COLORS['accent'],
            linewidth=1.2, transform=ax_kpi.transAxes
        )
        ax_kpi.add_patch(box)
        ax_kpi.text(0.5, y_top - 0.05, num, ha='center', va='top',
                    color=COLORS['accent'], fontsize=32, weight='bold',
                    family='monospace', transform=ax_kpi.transAxes)
        ax_kpi.text(0.5, y_top - 0.14, label, ha='center', va='top',
                    color=COLORS['text'], fontsize=10, weight='bold',
                    transform=ax_kpi.transAxes)
        ax_kpi.text(0.5, y_top - 0.18, sub, ha='center', va='top',
                    color=COLORS['muted'], fontsize=8, style='italic',
                    transform=ax_kpi.transAxes)
    ax_kpi.set_xlim(0, 1); ax_kpi.set_ylim(0, 1)

    ax_story = fig.add_subplot(gs[3, :]); ax_story.axis('off')
    ax_story.text(0.5, 0.85,
                  f'El capitán hizo su parte: {int(n_prog)} pases progresivos, {len(tiros)} tiros, {int(goles)} gol.',
                  ha='center', va='top', color=COLORS['text'],
                  fontsize=14, weight='bold', transform=ax_story.transAxes)
    ax_story.text(0.5, 0.45,
                  'Argentina perdió 1–2 contra todo pronóstico. El equipo, no.',
                  ha='center', va='top', color=COLORS['accent'],
                  fontsize=12, style='italic', transform=ax_story.transAxes)

    ax_foot = fig.add_subplot(gs[4, :]); ax_foot.axis('off')
    ax_foot.text(0.02, 0.5, '@datafutbol_ar', ha='left', va='center',
                 color=COLORS['accent'], fontsize=10, weight='bold',
                 transform=ax_foot.transAxes)
    ax_foot.text(0.98, 0.5, 'Datos: StatsBomb Open Data', ha='right', va='center',
                 color=COLORS['muted'], fontsize=9, style='italic',
                 transform=ax_foot.transAxes)

    guardar_fig(fig, archivo)
    plt.show()


dashboard_jugador(ev, 'Lionel Andrés Messi Cuccittini', 'punto5_dashboard_messi')


---

## Resumen — Punto 5 ✅

| Lo que pide la consigna | Cómo lo cumplo |
|---|---|
| Mapas de tiros y pases de ambos equipos | Dashboard A → fila 1 (shot maps) + fila 2 (pass maps) |
| Resultado y detalles del partido | Dashboard A → header con marcador + tabla 8 métricas |
| Datos sobre jugadores destacados (criterio propio) | Dashboard A → tarjetas con top destacado por equipo · Dashboard B → foco completo en Messi |

### Mi criterio editorial para "destacado"

Elegí **"jugador con más pases progresivos del partido"** como métrica para el Dashboard A en lugar de la opción típica (más recuperaciones, más toques). El motivo: los pases progresivos miden **intención ofensiva** y diferencian al jugador que "construye juego" del que "circula la pelota sin riesgo". Es la métrica que más se está usando en scouting profesional hoy.

Para el Dashboard B elegí a **Messi** por su importancia narrativa: capitán, único goleador de ARG, y la figura que da continuidad con el análisis del Punto 4.
